# TP — Implémentation d'un système RAG avec Langchain et Hugging Face

Ce notebook implémente un pipeline **RAG (Retrieval-Augmented Generation)** complet en 8 étapes :
1. Configuration de l'environnement
2. Chargement du jeu de données
3. Fractionnement des documents
4. Génération des embeddings
5. Création de l'entrepôt de vecteurs (FAISS)
6. Préparation du modèle LLM
7. Construction de la chaîne de questions-réponses
8. Test du système

> **Note sur les imports** : les classes `HuggingFaceDatasetLoader`, `HuggingFaceEmbeddings`, `FAISS`, `HuggingFacePipeline` et `RetrievalQA` ont été déplacées de `langchain` vers `langchain_community` / `langchain_huggingface` dans les versions récentes de Langchain. Ce notebook utilise donc les chemins d'import à jour pour garantir la compatibilité avec les bibliothèques installées.


## 1. Configuration de l'environnement

Chaque bibliothèque remplit un rôle précis :
- **langchain / langchain-community** : orchestration des composants du pipeline RAG
- **torch** : moteur de calcul pour les modèles de deep learning
- **transformers** : modèles pré-entraînés (embeddings, question-réponse)
- **sentence-transformers** : génération d'embeddings sémantiques
- **datasets** : chargement de jeux de données Hugging Face
- **faiss-cpu** : recherche de similarité vectorielle rapide


In [ ]:
!pip install -q langchain
!pip install -q langchain-community
!pip install -q langchain-huggingface
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu


## 2. Chargement du jeu de données

On utilise `HuggingFaceDatasetLoader` pour charger le jeu de données `databricks/databricks-dolly-15k` et transformer chaque exemple en `Document` Langchain, en se basant sur la colonne `context`.


In [ ]:
from langchain_community.document_loaders import HuggingFaceDatasetLoader

dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()

print(f"Nombre de documents chargés : {len(data)}")
print(data[:2])  # Aperçu des 2 premiers documents


## 3. Fractionnement des documents

Les LLM ont une limite de contexte. On découpe donc les documents en segments (`chunks`) de taille raisonnable, avec un léger chevauchement (`overlap`) pour préserver le contexte entre segments consécutifs.


In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)
docs = text_splitter.split_documents(data)

print(f"Nombre de segments (chunks) obtenus : {len(docs)}")
print(docs[0])  # Aperçu du premier segment


## 4. Génération des embeddings

Chaque segment de texte est converti en vecteur numérique via un modèle `sentence-transformers`, ce qui permet ensuite de mesurer la similarité sémantique entre une requête et les documents indexés.


In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings

modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device': 'cpu'}
encode_kwargs = {'normalize_embeddings': False}

embeddings = HuggingFaceEmbeddings(
    model_name=modelPath,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)


In [ ]:
# (Facultatif) Test de l'embedding sur une phrase
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])


## 5. Création de l'entrepôt de vecteurs (FAISS)

FAISS indexe les embeddings des segments de documents, ce qui permet des recherches de similarité rapides et évolutives lors des requêtes.


In [ ]:
from langchain_community.vectorstores import FAISS

# Cette étape peut prendre du temps selon la taille du jeu de données
db = FAISS.from_documents(docs, embeddings)
print("Index FAISS créé avec succès.")


## 6. Préparation du modèle LLM

On charge un modèle de question-réponse pré-entraîné (`Intel/dynamic_tinybert`) et on l'intègre dans un pipeline Hugging Face, puis on l'enveloppe dans un wrapper Langchain (`HuggingFacePipeline`) pour pouvoir l'utiliser dans la chaîne RAG.


In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_huggingface import HuggingFacePipeline

model_name = "Intel/dynamic_tinybert"

tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

question_answerer = pipeline(
    "question-answering",
    model=model,
    tokenizer=tokenizer,
    return_tensors='pt'
)

llm = HuggingFacePipeline(
    pipeline=question_answerer,
    model_kwargs={"temperature": 0.7, "max_length": 512},
)


## 7. Création de la chaîne de questions-réponses (RetrievalQA)

Cette chaîne relie le **retriever** (recherche des documents pertinents dans FAISS) au **LLM** (génération de la réponse à partir du contexte récupéré). C'est le cœur du pipeline RAG : Retrieve → Augment → Generate.


In [ ]:
from langchain.chains import RetrievalQA

retriever = db.as_retriever(search_kwargs={"k": 4})  # k = nombre de documents récupérés

qa = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="refine",
    retriever=retriever,
    return_source_documents=False
)


## 8. Test du système RAG

On soumet une question au pipeline complet pour vérifier que la récupération de documents et la génération de réponse fonctionnent correctement de bout en bout.


In [ ]:
question = "What is cheesemaking?"

result = qa.invoke({"query": question})
print(result["result"] if isinstance(result, dict) else result)


## Remarques et pistes d'amélioration

- Le modèle `Intel/dynamic_tinybert` est un modèle de type *extractive QA* : il extrait une réponse directement du contexte plutôt que de la générer librement. Pour des réponses plus fluides et génératives, on pourrait le remplacer par un modèle de type `text2text-generation` (ex. `google/flan-t5-base`) avec `pipeline("text2text-generation", ...)`.
- Le paramètre `k` du retriever (nombre de documents récupérés) peut être ajusté selon la richesse du contexte souhaité.
- Le `chain_type="refine"` traite les documents séquentiellement en affinant la réponse à chaque étape ; d'autres options existent (`stuff`, `map_reduce`, `map_rerank`) selon le compromis vitesse/qualité recherché.
